# Script 10b — Fine-tuning DistilGPT-2 on my Letterboxd reviews

We fine-tune a pre-trained language model on my translated reviews.
The model learns my writing style and can generate new reviews.

**Model:** DistilGPT-2 (82M parameters, runs well on CPU)  
**Data:** ~417 reviews translated to English  
**Goal:** given a film title and rating, generate a review in my style


## 1. Imports

In [1]:
import os, sys, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import torch
from transformers import (
    GPT2Tokenizer, GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    Trainer, TrainingArguments,
    pipeline
)
from datasets import Dataset

SCRIPT_DIR = os.path.join(os.getcwd(), '..')
DATA_DIR   = os.path.join(SCRIPT_DIR, 'data', 'processed')
OUT_DIR    = os.path.join(SCRIPT_DIR, 'output')
MODEL_DIR  = os.path.join(SCRIPT_DIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
print(f"PyTorch version: {torch.__version__}")


Device: cpu
PyTorch version: 2.11.0+cpu


## 2. Load and prepare the reviews

We format each review as a structured prompt:

```
<|film|> The Dark Knight (2008) <|rating|> 5.0 <|review|> ... <|endoftext|>
```

This teaches the model to associate film context with writing style.


In [2]:
df = pd.read_csv(os.path.join(DATA_DIR, 'reviews_translated.csv'))
df = df[df['translated_review'].notna()].copy()
df['translated_review'] = df['translated_review'].str.strip()
df = df[df['translated_review'].str.len() > 30]  # skip very short reviews

print(f"Reviews available: {len(df)}")
print(f"Avg length: {df['translated_review'].str.len().mean():.0f} chars")
print(f"Min length: {df['translated_review'].str.len().min()} chars")
print(f"Max length: {df['translated_review'].str.len().max()} chars")
print()

# Format each review as a structured prompt
def format_review(row):
    rating = f"{float(row['Rating']):.1f}" if pd.notna(row['Rating']) else "?"
    return (
        f"<|film|> {row['Name']} ({int(row['Year'])}) "
        f"<|rating|> {rating} "
        f"<|review|> {row['translated_review']} "
        f"<|endoftext|>"
    )

df['formatted'] = df.apply(format_review, axis=1)

print("Sample formatted review:")
print(df['formatted'].iloc[0][:300])


Reviews available: 425
Avg length: 635 chars
Min length: 32 chars
Max length: 2474 chars

Sample formatted review:
<|film|> Soul (2020) <|rating|> 4.5 <|review|> One of the most impactful films I've seen in recent times. And what I cried most about Pixar. Above all, it adds a much more sensitive, much more inspiring touch to an animated film. Something I haven't seen in a genre like this for a long time. <|endof


## 3. Load DistilGPT-2 tokenizer and model

**DistilGPT-2** is a distilled (compressed) version of GPT-2.
It's 40% smaller and 60% faster while retaining most of the quality.
Perfect for fine-tuning on CPU with a small dataset.


In [3]:
print("Loading DistilGPT-2...")

tokenizer = GPT2Tokenizer.from_pretrained('distilgpt2')

# Add our special tokens so the model learns the structure
special_tokens = {
    'pad_token': '<|pad|>',
    'additional_special_tokens': ['<|film|>', '<|rating|>', '<|review|>']
}
tokenizer.add_special_tokens(special_tokens)

model = GPT2LMHeadModel.from_pretrained('distilgpt2')
model.resize_token_embeddings(len(tokenizer))  # resize for new tokens

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {total_params/1e6:.0f}M parameters")
print(f"Vocabulary size: {len(tokenizer)}")


Loading DistilGPT-2...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model loaded: 82M parameters
Vocabulary size: 50261


## 4. Tokenise the dataset

We convert all text to token IDs (numbers) that the model understands.
We also split into train (90%) and validation (10%).


In [4]:
# Tokenise all reviews
def tokenise(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=256,        # max tokens per review
        padding='max_length',
    )

texts = df['formatted'].tolist()

# Train / validation split
split = int(len(texts) * 0.9)
train_texts = texts[:split]
val_texts   = texts[split:]

train_dataset = Dataset.from_dict({'text': train_texts}).map(tokenise, batched=True)
val_dataset   = Dataset.from_dict({'text': val_texts}).map(tokenise, batched=True)

# Set labels = input_ids for language modelling (predict next token)
train_dataset = train_dataset.map(lambda x: {'labels': x['input_ids']})
val_dataset   = val_dataset.map(lambda x: {'labels': x['input_ids']})

print(f"Training samples:   {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")


Map:   0%|          | 0/382 [00:00<?, ? examples/s]

Map:   0%|          | 0/43 [00:00<?, ? examples/s]

Map:   0%|          | 0/382 [00:00<?, ? examples/s]

Map:   0%|          | 0/43 [00:00<?, ? examples/s]

Training samples:   382
Validation samples: 43


## 5. Fine-tune the model

We train for 5 epochs — the model sees the full dataset 5 times.

**What is an epoch?**  
One complete pass through all training reviews.
More epochs = more learning, but risk of overfitting (memorising instead of learning).

This will take **20-40 minutes on CPU**. The loss should decrease each epoch.


In [5]:
training_args = TrainingArguments(
    output_dir=os.path.join(MODEL_DIR, 'checkpoints'),
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir=os.path.join(MODEL_DIR, 'logs'),
    logging_steps=20,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    report_to='none',
    use_cpu=True,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

trainer.train()


[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.551152,3.211902
2,3.129481,3.056866
3,2.898129,3.056515
4,2.738446,3.059597
5,2.786907,3.064594


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=480, training_loss=3.062328235308329, metrics={'train_runtime': 3108.5796, 'train_samples_per_second': 0.614, 'train_steps_per_second': 0.154, 'total_flos': 124769198407680.0, 'train_loss': 3.062328235308329, 'epoch': 5.0})

## 6. Save the model

In [6]:
model_path = os.path.join(MODEL_DIR, 'letterboxd_gpt2')
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)
print(f"Model saved to: {model_path}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: c:\Users\Huawei\Desktop\Private\Cinemaverse\scripts\..\models\letterboxd_gpt2


## 7. Generate reviews!

Now the fun part — give the model a film title and rating,
and let it write a review in your style.


In [7]:
def generate_review(film_name, year, rating, max_length=200, temperature=0.8):
    """
    Generate a review for a film using the fine-tuned model.
    
    temperature: controls randomness
      - low (0.5) = more conservative, repetitive
      - high (1.2) = more creative, sometimes incoherent
    """
    prompt = f"<|film|> {film_name} ({year}) <|rating|> {rating:.1f} <|review|>"
    
    inputs = tokenizer.encode(prompt, return_tensors='pt')
    
    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_length=max_length,
            temperature=temperature,
            do_sample=True,
            top_k=50,           # only consider top 50 tokens
            top_p=0.95,         # nucleus sampling
            repetition_penalty=1.2,  # penalise repeating phrases
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    generated = tokenizer.decode(outputs[0], skip_special_tokens=False)
    
    # Extract just the review part
    if '<|review|>' in generated:
        review = generated.split('<|review|>')[1]
        review = review.replace('<|endoftext|>', '').strip()
        return review
    return generated

# Test with a few films
test_films = [
    ("Oppenheimer", 2023, 5.0),
    ("Madame Web",  2024, 0.5),
    ("Dune: Part Two", 2024, 5.0),
]

for film, year, rating in test_films:
    print(f"Film: {film} ({year}) — Rating: {rating}")
    print("-" * 50)
    review = generate_review(film, year, rating)
    print(review)
    print()


Film: Oppenheimer (2023) — Rating: 5.0
--------------------------------------------------
I'm going to miss the film completely, because in a movie with such low expectations...
This is probably one of those movies that will never make me feel like they've managed it properly — only time can tell if everything's fine and everyone has been well prepared for this journey… And how quickly was "Dangerous" realized? They were so good at playing characters capable of destroying anything even though things weren't exactly perfect yet!? It could have gone something else without mentioning them again: THE PICKER OF FIFTH DOUGHT AT YUM!

Film: Madame Web (2024) — Rating: 0.5
--------------------------------------------------
It’s a bit unsettling to say the least: this is my first film with such strong scenes, I still have moments where you get confused about some of your favorite characters and it feels unnecessary for me just yet!
It was completely predictable — at best; but once again… The ci

## 8. Batch generate for watchlist recommendations

Generate reviews for the top recommended films from script 08b.


In [8]:
import json

# Load top watchlist recommendations
try:
    recs = pd.read_csv(os.path.join(DATA_DIR, 'recs_watchlist.csv')).head(10)
    
    generated_reviews = []
    for _, row in recs.iterrows():
        film   = row['Name']
        year   = int(row['Year'])
        # Use predicted rating from script 09 if available, else similarity score
        rating = float(row.get('predicted_rating', 4.0))
        
        print(f"Generating: {film} ({year}) — {rating:.1f}*")
        review = generate_review(film, year, rating, temperature=0.85)
        generated_reviews.append({
            'Name':             film,
            'Year':             year,
            'predicted_rating': rating,
            'generated_review': review,
        })
        print(f"  {review[:120]}...")
        print()
    
    # Save
    pd.DataFrame(generated_reviews).to_csv(
        os.path.join(DATA_DIR, 'generated_reviews.csv'),
        index=False, encoding='utf-8'
    )
    print("Generated reviews saved to data/processed/generated_reviews.csv")

except FileNotFoundError:
    print("Run script 08b first to generate watchlist recommendations.")
    print("Then re-run this cell.")


Generating: The Good, the Bad and the Ugly (1966) — 4.0*
  A few short films left me with a lot of fun...
Some genuinely incredible moments were made even more difficult by Oscar...

Generating: Persona (1966) — 4.0*
  I was expecting it to be a decent film… But seriously, the first part is quite entertaining—and honestly, in that regard...

Generating: A Patch of Blue (1965) — 4.0*
  This is a masterpiece, and with plenty of positives I'm still convinced that it's very good to see the film in its own w...

Generating: High and Low (1963) — 4.0*
  I’ve never seen a film with such low budget projects, that you can easily see why they have so few ideas: the actors wer...

Generating: Serpico (1973) — 4.0*
  I'm a fan of horror, but when the actors start playing characters they end up thinking "Oh… Oh... Yay." And how could we...

Generating: Kes (1969) — 4.0*
  Another movie I’m not sure of… A cult film that ends with a few minutes left...
It started off by saying what the hell ...

Gene

## 9. Experiment — adjust temperature

Try different temperatures to see how they affect the output.
Lower = more like your writing. Higher = more creative but less coherent.


In [9]:
film, year, rating = "The Substance", 2024, 4.0

print(f"Film: {film} ({year}) — Rating: {rating}")
print()

for temp in [0.5, 0.8, 1.1]:
    print(f"Temperature = {temp}:")
    print("-" * 40)
    review = generate_review(film, year, rating, temperature=temp)
    print(review[:200])
    print()


Film: The Substance (2024) — Rating: 4.0

Temperature = 0.5:
----------------------------------------
I’m not going to lie, this is a horror film with such an incredible plot and the ending itself could easily be explained by many scenes that left me speechless: “I don't want you dead… You deserve mor

Temperature = 0.8:
----------------------------------------
It's not all that simple, but it sure doesn't change the way people think about this film: At times we just get scared of what is happening in our lives and are genuinely afraid they can be seen watch

Temperature = 1.1:
----------------------------------------
Not surprisingly, this soundtrack was so incredibly solid I’ve been trying to stay tuned for a few months now... It left my ears racing like crazy when the film started playing again… In fact it all f



## Summary

**What we built:**
- Fine-tuned DistilGPT-2 on 417 Letterboxd reviews
- Model learns writing style, tone and vocabulary
- Can generate new reviews given a film title + rating

**Limitations:**
- 417 reviews is a small dataset — the model captures style patterns but isn't perfect
- Sometimes repeats phrases or loses coherence mid-sentence
- Works best for films/genres similar to what's in the training data

**The model is saved in `models/letterboxd_gpt2/`** — it can be loaded and used in the final dashboard.
